# Eutectic Si specialist 학습

GroupNorm U-Net 이진 분할. Patch 384, 원본당 8 crop × 회전 4, BCE + Dice.


## 설정


In [ ]:
from pathlib import Path
from google.colab import drive
import os
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/KoMaP2026")
TARGET_CLASS = 2
RUN_NAME = 'specialist_eutectic_fast_01'
EPOCHS = 90
BATCH_SIZE = 4
PATCH_SIZE = 384
STRIDE = 256
BASE_PATCHES_PER_IMAGE = 8
LEARNING_RATE = 1e-3
EVAL_EVERY = 5
SEED = 42
GAMMA_PROBABILITY = 1.0
GAMMA_MIN, GAMMA_MAX = 0.80, 1.60
NUM_WORKERS = min(2, os.cpu_count() or 1)
PREFETCH_FACTOR = 2
PROGRESS_EVERY = 100

DATA_ROOT = Path("/content/komap_specialists_data")
RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
CODE_DIR = CODE_ROOT / '.'
sys.dont_write_bytecode = True

for _module_name in list(sys.modules):
    if _module_name.startswith('komap_'):
        del sys.modules[_module_name]
sys.path[:] = [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print('제출 코드:', CODE_DIR)


## 데이터·초기 모델


In [ ]:
import komap_specialist_data as kd
import komap_specialist as sp
for module in (kd, sp):
    importlib.reload(module)
    if Path(module.__file__).resolve().parent != CODE_DIR.resolve():
        raise RuntimeError("다른 코드가 로드됐습니다. 런타임을 다시 시작하세요.")

CONFIG = sp.Config(target_class=TARGET_CLASS, epochs=EPOCHS, batch_size=BATCH_SIZE,
                   patch_size=PATCH_SIZE, stride=STRIDE, samples_per_image=BASE_PATCHES_PER_IMAGE,
                   lr=LEARNING_RATE, eval_every=EVAL_EVERY, seed=SEED,
                   gamma_probability=GAMMA_PROBABILITY, gamma_min=GAMMA_MIN, gamma_max=GAMMA_MAX)
RUNTIME = sp.RuntimeOptions(num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR,
                            progress_every=PROGRESS_EVERY)
RUNTIME.validate()
DATA_REPORT = sp.prepare(PROJECT_ROOT, DATA_ROOT, CONFIG)
print("저장 폴더:", RUN_DIR)


## 학습·검증


In [ ]:
BEST_PATH = sp.train(DATA_ROOT, RUN_DIR, CONFIG, RUNTIME)
print("best:", BEST_PATH)


## 확률 내보내기


In [ ]:
EXPORT_DIR = sp.export_best(RUN_DIR, DATA_ROOT)
print("확률 맵:", EXPORT_DIR)
print("결합 노트북이 읽는 파일:", RUN_DIR / "export_index.json")
